In [ ]:
# Run this notebook from Mini_project_4 after installing its dependencies.
%pip install -r requirements.txt


In [ ]:
# Dependencies are installed once in the first cell.


In [ ]:
# Dependencies are installed once in the first cell.


In [ ]:
import torch

# if gpu is to be used, this will help the system so i added it to my code
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
# Dependencies are installed once in the first cell.


In [ ]:
import io
import os
import glob
import torch
import base64
import stable_baselines3
import numpy as np
import matplotlib.pyplot as plt
from stable_baselines3 import DQN
from stable_baselines3.common.results_plotter import ts2xy, load_results
from stable_baselines3.common.callbacks import EvalCallback
from stable_baselines3.common.env_util import make_atari_env
import gymnasium as gym
from gymnasium import spaces
from gymnasium.wrappers import RecordVideo
from IPython.display import HTML
from IPython import display
from IPython import display as ipythondisplay
from IPython import display as ipythondisplay
import random
from collections import namedtuple, deque
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

In [ ]:
# Dependencies are installed once in the first cell.


In [ ]:
# We should import gym library here
import gymnasium as gym

# Initialize Lunar Lander environment with RGB rendering mode
env = gym.make('LunarLander-v3', render_mode="rgb_array")

# Find observation size (state size)
state_size = env.observation_space.shape[0]

# Find action size, We can see it blew:
# 0: Do nothing
# 1: Fire left engine
# 2: Fire down engine
# 3: Fire right engine
action_size = env.action_space.n

env.close()
state_size, action_size

In [ ]:
import glob
import io
import base64
from IPython.display import HTML, display as ipythondisplay

# RGB rendering does not require starting an Xvfb process.

"""
Utility functions to enable video recording of gym environment
and displaying it.
To enable video, just do "env = wrap_env(env)"
"""

def show_video():
    mp4list = glob.glob('video/*.mp4')
    if len(mp4list) > 0:
        mp4 = mp4list[0]
        video = io.open(mp4, 'r+b').read()
        encoded = base64.b64encode(video)
        ipythondisplay(HTML(data=f'''<video alt="test" autoplay loop controls style="height: 400px;">
                  <source src="data:video/mp4;base64,{encoded.decode('ascii')}" type="video/mp4" />
                </video>'''))
    else:
        print("Could not find video")

In [ ]:
import random
from collections import namedtuple, deque

# Define the transition tuple
Transition = namedtuple('Transition', ('state', 'action', 'next_state', 'reward', 'done'))

class ExperienceReplay:
    def __init__(self, capacity):
        self.memory = deque(maxlen=capacity)

    def store_transition(self, state, action, next_state, reward, done):
        """Store a transition in the replay buffer."""
        transition = Transition(state, action, next_state, reward, done)
        self.memory.append(transition)

    def sample(self, batch_size):
        """Sample a batch of transitions from the replay buffer."""
        return random.sample(list(self.memory), batch_size)

    def __len__(self):
        """Return the current size of internal memory."""
        return len(self.memory)


In [ ]:
import torch.nn as nn
import torch.nn.functional as F

class DeepQNetwork(nn.Module):
    def __init__(self, state_size, action_size):
        """
        Initialize the Deep Q-Network.

        Args:
            state_size (int): The size of the input state.
            action_size (int): The size of the action space.
        """
        super(DeepQNetwork, self).__init__()

        self.net = nn.Sequential(
            nn.Linear(state_size, 512),  # First hidden layer
            nn.ReLU(),                   # Activation function
            nn.LayerNorm(512),           # Layer normalization
            nn.Dropout(0.1),             # Dropout for regularization
            nn.Linear(512, 512),         # Second hidden layer
            nn.ReLU(),                   # Activation function
            nn.LayerNorm(512),           # Layer normalization
            nn.Dropout(0.1),             # Dropout for regularization
            nn.Linear(512, 512),         # Third hidden layer
            nn.ReLU(),                   # Activation function
            nn.Linear(512, action_size)  # Output layer
        )

    def forward(self, x):
        """
        Perform a forward pass through the network.

        Args:
            x (torch.Tensor): The input state tensor.

        Returns:
            torch.Tensor: The output Q-values for each action.
        """
        return self.net(x)

# DQN

In [ ]:
# DQN agent
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

In [ ]:
import torch
import torch.optim as optim
import numpy as np

class DQNAgent:
    def __init__(self, state_size, action_size, batch_size, gamma=0.99, buffer_size=25000, alpha=1e-4, tau=1e-3):
        """
        Initialize the DQNAgent.

        Args:
            state_size (int): Dimension of each state.
            action_size (int): Dimension of each action.
            batch_size (int): Size of each training batch.
            gamma (float): Discount factor.
            buffer_size (int): Replay buffer size.
            alpha (float): Learning rate.
            tau (float): For soft update of target network parameters.
        """
        self.state_size = state_size
        self.action_size = action_size
        self.batch_size = batch_size
        self.gamma = gamma
        self.tau = tau  # For soft update of target network parameters
        self.experience_replay = ExperienceReplay(buffer_size)

        # Main network
        self.value_net = DeepQNetwork(state_size, action_size).to(device)
        # Target network
        self.target_net = DeepQNetwork(state_size, action_size).to(device)
        self.target_net.load_state_dict(self.value_net.state_dict())
        self.target_net.eval()  # Target network does not train

        self.optimizer = optim.Adam(self.value_net.parameters(), lr=alpha)

    def take_action(self, state, eps=0.0):
        """
        Select an action using an epsilon-greedy policy.

        Args:
            state (array): Current state.
            eps (float): Epsilon, for exploration-exploitation trade-off.

        Returns:
            int: Selected action.
        """
        self.value_net.eval()
        if random.random() > eps:
            with torch.no_grad():
                state_tensor = torch.tensor(state).float().unsqueeze(0).to(device)
                action = torch.argmax(self.value_net(state_tensor)).item()
        else:
            action = np.random.randint(0, self.action_size)
        self.value_net.train()
        return action

    def update_params(self):
        """
        Update the parameters of the value network using experience replay and
        perform a soft update of the target network parameters.
        """
        if len(self.experience_replay) < self.batch_size:
            return
        batch = Transition(*zip(*self.experience_replay.sample(self.batch_size)))

        state_batch = torch.tensor(np.array(batch.state), dtype=torch.float32).to(device)
        action_batch = torch.tensor(batch.action, dtype=torch.int64).unsqueeze(1).to(device)
        next_state_batch = torch.tensor(np.array(batch.next_state), dtype=torch.float32).to(device)
        reward_batch = torch.tensor(batch.reward, dtype=torch.float32).unsqueeze(1).to(device)
        done_batch = torch.tensor(batch.done, dtype=torch.float32).unsqueeze(1).to(device)

        q_expected = self.value_net(state_batch).gather(1, action_batch)
        q_targets_next = self.target_net(next_state_batch).detach().max(1)[0].unsqueeze(1)
        q_targets = reward_batch + (self.gamma * q_targets_next * (1 - done_batch))
        loss = F.mse_loss(q_expected, q_targets)

        self.optimizer.zero_grad()
        loss.backward()
        self.optimizer.step()

        # Soft update target network parameters
        for target_param, local_param in zip(self.target_net.parameters(), self.value_net.parameters()):
            target_param.data.copy_(self.tau * local_param.data + (1.0 - self.tau) * target_param.data)

    def save(self, fname):
        """
        Save the model parameters.

        Args:
            fname (str): Filename to save the model.
        """
        torch.save(self.value_net.state_dict(), fname)

    def load(self, fname):
        """
        Load the model parameters.

        Args:
            fname (str): Filename to load the model.
        """
        self.value_net.load_state_dict(torch.load(fname, map_location=device, weights_only=True))
        self.target_net.load_state_dict(torch.load(fname, map_location=device, weights_only=True))


In [ ]:
import torch
import torch.optim as optim
import numpy as np

# Define constants
n_episodes = 250
eps = 1.0
eps_decay_rate = 0.97
eps_end = 0.01
BATCH_SIZE = 32

In [ ]:
# Dependencies are installed once in the first cell.


In [ ]:
# training phase

# Mine
agent = DQNAgent(state_size, action_size, batch_size=BATCH_SIZE)
crs = np.zeros(n_episodes)
crs_recent = deque(maxlen=25)

for i_episode in range(1, n_episodes + 1):
    if i_episode % 50 == 0:
        env = RecordVideo(gym.make("LunarLander-v3", render_mode="rgb_array"), f"./DQN/batch{BATCH_SIZE}/eps{i_episode}", episode_trigger=lambda episode: True, video_length=0)
    else:
        env = gym.make("LunarLander-v3", render_mode="rgb_array")

    try:
        state, info = env.reset()
        done = False
        cr = 0

        while not done:
            action = agent.take_action(state, eps)
            next_state, reward, terminated, truncated, info = env.step(action)
            done = terminated or truncated
            agent.experience_replay.store_transition(state, action, next_state, reward, terminated)
            agent.update_params()
            state = next_state
            cr += reward
    finally:
        env.close()

    # Decay epsilon
    eps = max(eps * eps_decay_rate, eps_end)
    crs[i_episode - 1] = cr
    crs_recent.append(cr)

    # Save model every 50 episodes
    if i_episode % 50 == 0:
        agent.save(f"q_net_batch{BATCH_SIZE}_eps{i_episode}.pt")

    # Print average reward every 25 episodes
    print(f'\rEpisode {i_episode}\tAverage Reward: {np.mean(crs_recent):.2f}\tEpsilon: {eps:.2f}', end="")
    if i_episode % 25 == 0:
        print(f'\rEpisode {i_episode}\tAverage Reward: {np.mean(crs_recent):.2f}\tEpsilon: {eps:.2f}')

# Each episode closes its environment in the finally block above.
